In [ ]:
import json
import math
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
with open("../../data/scenario_cumulative_returns.json") as f:
    scenarios = json.load(f) #nested dict, scenarios[scenario][ticker] = cumulative multiplier

In [ ]:
def calc_cvar(weights, scenarios, alpha, weight_tolerance=1e-6):
    """
    Conditional Value-at-Risk (CVaR) of a given portfolio, using the scenario
    cumulative returns. Same logic as the solver in main_solver.ipynb.

    Args:
        weights: Dict {ticker: weight}, e.g. the "weights" entry of solver.output.json
        scenarios: Dict of scenarios with cumulative returns, keyed by scenario ID
                   e.g., {'0': {'TICKER': multiplier, ...}, '1': {...}, ...}
        alpha: CVaR confidence level (e.g., 0.95 for 95% CVaR)
        weight_tolerance: Allowed deviation of the weights' sum from 1

    Returns:
        Tuple: (cvar, gamma) where gamma is the empirical VaR (the alpha-quantile of the losses)
    """
    S = len(scenarios)

    # Fail loudly on bad input
    if abs(sum(weights.values()) - 1) > weight_tolerance:
        raise ValueError(f"Weights must sum to 1, got {sum(weights.values())}")
    first_scenario = next(iter(scenarios.values()))
    missing = [t for t in weights if t not in first_scenario]
    if missing:
        raise KeyError(f"Tickers not found in scenarios: {missing}")

    # Loss per scenario: 1 - portfolio cumulative return
    losses = np.array([
        1 - sum(w * scenarios[s_key][ticker] for ticker, w in weights.items())
        for s_key in scenarios
    ])

    # gamma = VaR: the loss at position ceil(alpha * S) in the sorted losses
    # (the small offset guards against floating point noise in alpha * S)
    k = math.ceil(alpha * S - 1e-9)
    gamma = float(np.sort(losses)[k - 1])

    # Same objective as the solver: gamma + 1/((1-alpha)S) * sum of loss overage
    cvar = gamma + np.sum(np.maximum(losses - gamma, 0)) / ((1 - alpha) * S)

    return float(cvar), gamma

## Check against the solver

Recompute CVaR for stored portfolios from solver.output.json (lowest, middle and highest target return). It should match the stored values.

In [ ]:
with open("../../data/solver.output.json") as f:
    solver_output = json.load(f)

results = solver_output["results"]
alpha = solver_output["summary"]["alpha"]

for p in (results[0], results[len(results) // 2], results[-1]):
    cvar, gamma = calc_cvar(p["weights"], scenarios, alpha)
    print(f"target_return={p['target_return']:.4f}  "
          f"cvar: calc={cvar:.6f} solver={p['cvar']:.6f}  "
          f"gamma: calc={gamma:.6f} solver={p['gamma']:.6f}")
    assert abs(cvar - p["cvar"]) < 1e-6
    assert abs(gamma - p["gamma"]) < 1e-6

## Equal-weight portfolio

CVaR of a portfolio that holds every asset with weight 1/n.

In [ ]:
tickers = list(scenarios["0"].keys())
n = len(tickers)
equal_weights = {ticker: 1 / n for ticker in tickers}

cvar_equal, gamma_equal = calc_cvar(equal_weights, scenarios, alpha)
print(f"Equal weights: {n} assets, weight per asset = {1 / n:.6f}")
print(f"alpha={alpha}  CVaR={cvar_equal:.6f}  gamma (VaR)={gamma_equal:.6f}")

## Variance comparison

In [ ]:
with open("../../data/expected_returns_covariance_matrix.json") as f:
    cov_data = json.load(f) #keys: tickers, expected_returns, covariance_matrix (weekly)

In [ ]:
def calc_variance(weights, cov_data, weight_tolerance=1e-6):
    """
    Variance of a given portfolio: w^T Sigma w.

    Args:
        weights: Dict {ticker: weight}, e.g. the "weights" entry of solver.output.json
        cov_data: Dict loaded from expected_returns_covariance_matrix.json, with the keys
                  "tickers" (list) and "covariance_matrix" (nested list, same order as tickers)
        weight_tolerance: Allowed deviation of the weights' sum from 1

    Returns:
        Portfolio variance, in the same time unit as the covariance matrix (weekly)
    """
    tickers = cov_data["tickers"]
    cov_matrix = np.array(cov_data["covariance_matrix"])

    # Fail loudly on bad input
    if abs(sum(weights.values()) - 1) > weight_tolerance:
        raise ValueError(f"Weights must sum to 1, got {sum(weights.values())}")
    missing = [t for t in weights if t not in tickers]
    if missing:
        raise KeyError(f"Tickers not found in covariance data: {missing}")

    # Weight vector in the same order as the covariance matrix (0 for tickers not in weights)
    w = np.array([weights.get(ticker, 0.0) for ticker in tickers])

    return float(w @ cov_matrix @ w)

In [ ]:
# Single asset: variance must equal that ticker's diagonal entry
test_idx = 0
test_ticker = cov_data["tickers"][test_idx]
var_single = calc_variance({test_ticker: 1.0}, cov_data)
print(f"{test_ticker}: calc={var_single:.8f} diagonal={cov_data['covariance_matrix'][test_idx][test_idx]:.8f}")
assert abs(var_single - cov_data["covariance_matrix"][test_idx][test_idx]) < 1e-12

# Unknown ticker must raise
try:
    calc_variance({"NOT_A_TICKER": 1.0}, cov_data)
    raise AssertionError("expected a KeyError for an unknown ticker")
except KeyError as e:
    print(f"Unknown ticker raises KeyError as expected: {e}")

In [ ]:
# Same equal-weight portfolio as in the CVaR section above
var_equal_weekly = calc_variance(equal_weights, cov_data)

print(f"Equal weights: {len(equal_weights)} assets, weight per asset = {1 / len(equal_weights):.6f}")
print(f"Variance (weekly)    = {var_equal_weekly:.8f}")

Equal weights: 130 assets, weight per asset = 0.007692
Variance (weekly)    = 0.00059164


In [ ]:
def plot_variance_frontier(results, cov_data, equal_weight_variance, candidate_range=None,
                           save_path="../../data/variance_frontier.png"):
    """
    Plot the solver portfolios as weekly variance (x) against target return (y),
    with a red vertical line at the weekly variance of the equal-weight portfolio.
    The figure is saved as a PNG to save_path.

    Args:
        results: List of portfolios from solver.output.json ("results"), each with
                 "target_return" and "weights"
        cov_data: Dict loaded from expected_returns_covariance_matrix.json
        equal_weight_variance: Weekly variance of the equal-weight portfolio
        candidate_range: Optional (min, max) target return, inclusive. Portfolios in this
                         range are marked as candidate portfolios in green
        save_path: Where to save the PNG (None = do not save)

    Returns:
        Tuple: (fig, ax)
    """
    target_returns = [p["target_return"] for p in results]
    variances = [calc_variance(p["weights"], cov_data) for p in results]

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.plot(variances, target_returns, 'o-', linewidth=2.5, markersize=5,
            label='Solver portfolios', color='#1f77b4', alpha=0.8)

    if candidate_range is not None:
        low, high = candidate_range
        tol = 1e-9  # the grid targets carry floating point noise
        candidates = [p for p in results if low - tol <= p["target_return"] <= high + tol]
        if not candidates:
            raise ValueError(f"No solver portfolios with target return in [{low}, {high}]")
        ax.plot([calc_variance(p["weights"], cov_data) for p in candidates],
                [p["target_return"] for p in candidates],
                'o', markersize=11, color='#2ca02c', markeredgecolor='black',
                markeredgewidth=1.5, label=f'Candidate portfolios ({low} to {high})', zorder=5)

    ax.axvline(equal_weight_variance, color='red', linewidth=2, linestyle='-',
               label=f'Equal-weight portfolio (variance = {equal_weight_variance:.6f})')

    ax.set_xlabel('Portfolio variance (weekly)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Target Return (annual multiplier)', fontsize=14, fontweight='bold')
    ax.set_title('Solver Portfolios: Weekly Variance vs Target Return',
                 fontsize=16, fontweight='bold', pad=20)

    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=12, loc='best', framealpha=0.95)
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('white')

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return fig, ax

In [ ]:
fig, ax = plot_variance_frontier(solver_output["results"], cov_data, var_equal_weekly,
                                 candidate_range=(1.16, 1.22))